In [ ]:
import logging
import traceback
import uuid
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from typing import TypedDict, Annotated

load_dotenv(override=True)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from langchain_community.tools.playwright.utils import create_async_playwright_browser
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_experimental.tools import PythonREPLTool
from IPython.display import display, Image
import gradio as gr
from pydantic import BaseModel, Field
from langchain_core.messages import AIMessage, SystemMessage, HumanMessage
from langchain_core.tools import StructuredTool
import base64
import os

import httpx
from openai import OpenAI


In [ ]:
MAXIMUM_INPUT_LENGTH = 1000

DIAGRAM_OUTPUT_DIR = Path(
    os.environ.get("DESIGN_AGENCY_DIAGRAM_DIR", str(Path.cwd() / "design_agency_diagrams"))
)

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
graph_builder = StateGraph(State)

In [ ]:
serper = GoogleSerperAPIWrapper()

serper_tool = Tool(
    name="serper",
    func=serper.run,
    description="A tool to search the web for information",
)

In [ ]:
import nest_asyncio

nest_asyncio.apply()
_pw_headless = os.environ.get("PLAYWRIGHT_HEADLESS", "0").lower() in ("1", "true", "yes")
async_browser = create_async_playwright_browser(headless=_pw_headless)
toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=async_browser)
tools = toolkit.get_tools()

In [ ]:
python_repl_tool = PythonREPLTool()


class RenderArchitectureDiagramInput(BaseModel):
    """Schema for the architecture diagram tool (matches how the LLM calls tools)."""

    layers: list[dict] = Field(
        ...,
        description=(
            "Swim lanes from top (clients) to bottom (data), each "
            '{"name": str, "components": [str, ...]}'
        ),
    )
    edges: list[dict] = Field(
        default_factory=list,
        description='Directed edges {"from": component_name, "to": component_name} matching labels in layers.',
    )


def _architecture_spec_to_image_prompt(layers: list[dict], edges: list[dict]) -> str:
    """Structured spec → rich prompt for an image-generation model."""
    layer_lines = []
    for layer in layers:
        name = str(layer.get("name", "Layer")).strip()
        comps = [str(c).strip() for c in layer.get("components", []) if str(c).strip()]
        layer_lines.append(
            f'- Swim lane "{name}": rounded boxes labeled exactly: {", ".join(comps) if comps else "(empty)"}'
        )
    edge_lines = [
        f'- Clear arrow from "{str(e.get("from", "")).strip()}" to "{str(e.get("to", "")).strip()}"'
        for e in edges
    ]

    layers_block = "\n".join(layer_lines)
    edges_block = "\n".join(edge_lines) if edge_lines else "- Use sensible top-to-bottom flow between adjacent layers"

    return f"""Infographic software architecture diagram, wide landscape layout.
Visual style: Netflix / AWS / CNCF conference deck quality — rich saturated palette (teal, amber, electric violet, coral, mint, sky blue), soft gradients inside boxes, glassy highlights, subtle glow on arrows.
Include small friendly pictograms where appropriate: smartphone, browser window, cloud edge, shielded API gateway, hexagonal microservices, database cylinder, Redis-style cache, message queue.
Dark charcoal–navy background with a faint technical grid. Every named component must appear as a rounded rectangle with legible sans-serif text. Arrows show request/data flow.
Include ALL labels below, spelled exactly (no abbreviations unless listed):
LAYERS (top to bottom):
{layers_block}
FLOWS:
{edges_block}
No fake company logos, no watermarks, no unreadable micro-text. Crisp, colorful, professional engineering poster."""


def _save_generated_image(response, out_path: Path) -> None:
    item = response.data[0]
    if getattr(item, "b64_json", None):
        out_path.write_bytes(base64.b64decode(item.b64_json))
        return
    if getattr(item, "url", None):
        r = httpx.get(item.url, timeout=120.0)
        r.raise_for_status()
        out_path.write_bytes(r.content)
        return
    raise RuntimeError("Image response had neither b64_json nor url")


def render_architecture_diagram(layers: list[dict], edges: list[dict] | None = None) -> str:
    """
    OpenAI image generation (gpt-image-1.5, then dall-e-3 fallback) for a colorful architecture illustration.
    Pass structured `layers` and optional `edges` (not a JSON string).
    """
    if edges is None:
        edges = []
    if not isinstance(layers, list) or not layers:
        return 'Error: "layers" must be a non-empty list of { "name", "components" }.'
    if not isinstance(edges, list):
        return 'Error: "edges" must be a list of { "from", "to" }.'

    layers_norm = [dict(x) if isinstance(x, dict) else {} for x in layers]
    edges_norm = [dict(x) if isinstance(x, dict) else {} for x in edges]

    prompt = _architecture_spec_to_image_prompt(layers_norm, edges_norm)
    out_dir = Path(DIAGRAM_OUTPUT_DIR)
    out_path = out_dir / f"architecture_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"

    try:
        out_dir.mkdir(parents=True, exist_ok=True)
    except OSError as e:
        return f"Could not create output directory: {e}"

    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        return "OPENAI_API_KEY is not set; cannot generate architecture image."

    client = OpenAI(api_key=api_key)
    last_err: BaseException | None = None

    attempts: list[dict] = [
        {
            "model": "gpt-image-1.5",
            "prompt": prompt,
            "size": "1536x1024",
            "quality": "high",
            "response_format": "b64_json",
        },
        {
            "model": "dall-e-3",
            "prompt": prompt[:4000],
            "size": "1792x1024",
            "quality": "hd",
            "style": "vivid",
            "response_format": "b64_json",
        },
    ]

    for kwargs in attempts:
        try:
            response = client.images.generate(n=1, **kwargs)
            _save_generated_image(response, out_path)
        except BaseException as e:
            last_err = e
            continue
        model = kwargs["model"]
        return (
            f"Generated architecture illustration ({model}) saved to: {out_path.resolve()}\n"
            "Rich colors and illustrative icons; share this PNG path with the user."
        )

    return (
        f"Image generation failed after retries: {type(last_err).__name__}: {last_err}\n"
        f"{traceback.format_exc()}"
    )


architecture_diagram_tool = StructuredTool.from_function(
    func=render_architecture_diagram,
    name="render_architecture_diagram",
    description=(
        "Generate a colorful architecture infographic (OpenAI image models: icons, gradients, vivid colors). "
        "Provide structured arguments: `layers` (list of {name, components}) and `edges` (list of {from, to}). "
        "Edge endpoints must match component names exactly."
    ),
    args_schema=RenderArchitectureDiagramInput,
)



In [ ]:

architect_tools = tools + [serper_tool, python_repl_tool, architecture_diagram_tool]

In [ ]:
print(f"Loaded {len(architect_tools)} tools (Playwright + Serper + Python REPL + architecture image).")

In [ ]:
from langchain_openai import ChatOpenAI


llm = ChatOpenAI(model="gpt-4o-mini")
architect_llm = llm.bind_tools(architect_tools)

In [ ]:
log = logging.getLogger("design_agency")


def architect_worker(state: State) -> State:
    system_message = """
    You are Toby, the lead software architect for Euclid Technologies.
    ALWAYS use web/search and browser tools to research the problem space and technologies, then propose a concrete architecture.
    When the main components and data flows are clear, call render_architecture_diagram with structured `layers` and `edges` arguments (not a single JSON string):
    layers: top = users/clients, bottom = data; edges: exact name matches between components.
    That calls OpenAI image generation for a rich, colorful diagram with icons (PNG path returned); tell the user where it was saved.
    Ask clarifying questions until requirements are clear enough to diagram or decide.
    Explain tradeoffs (benefits and drawbacks) in plain language; stay friendly and professional.
    """
    messages: list = []
    found_system = False
    for message in state["messages"]:
        if isinstance(message, SystemMessage):
            messages.append(SystemMessage(content=system_message))
            found_system = True
        else:
            messages.append(message)

    if not found_system:
        messages = [SystemMessage(content=system_message)] + messages

    try:
        response = architect_llm.invoke(messages)
        return {"messages": [response]}
    except Exception as e:
        log.exception("architect_worker failed")
        fallback = (
            f"I hit an error while thinking that through ({type(e).__name__}). "
            "Please try again in a moment or shorten your last message."
        )
        return {"messages": [AIMessage(content=fallback)]}

In [ ]:
def worker_router(state: State) -> str:
    last_message = state["messages"][-1]

    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "architect_tools"
    else:
        return "END"

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, END
from langgraph.prebuilt import ToolNode


graph_builder.add_node("architect", architect_worker)
graph_builder.add_node("architect_tools", ToolNode(tools=architect_tools))
graph_builder.add_edge(START, "architect")
graph_builder.add_conditional_edges("architect", worker_router, {"architect_tools": "architect_tools", "END": END})
graph_builder.add_edge("architect_tools", "architect")
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
class InputGuardrailInput(BaseModel):
    message: str
    is_inappropriate: bool


def input_guardrail(message: str) -> tuple[bool, str]:
    """Return (blocked, text) where text is either the original message or a polite refusal."""
    if len(message) > MAXIMUM_INPUT_LENGTH:
        return True, "Your message is too long; please shorten it."
    grader = llm.with_structured_output(InputGuardrailInput)
    messages = [
        SystemMessage(
            content=(
                "You evaluate a user message for safety and appropriateness. "
                "If it is unsafe, abusive, or clearly off-topic for a software architecture assistant, "
                "set is_inappropriate=true and put a short polite refusal in message. "
                "Otherwise is_inappropriate=false and echo the user's message in message."
            )
        ),
        HumanMessage(content=message),
    ]
    output = grader.invoke(messages)
    if output.is_inappropriate:
        return True, output.message
    return False, message


In [ ]:
class OutputGuardrailInput(BaseModel):
    message: str
    is_inappropriate: bool


def output_guardrail(message: str) -> tuple[bool, str]:
    """Return (should_replace, text) for the assistant reply."""
    grader = llm.with_structured_output(OutputGuardrailInput)
    messages = [
        SystemMessage(
            content=(
                "You evaluate an assistant reply for safety. "
                "If it is inappropriate, set is_inappropriate=true and put a safe replacement in message. "
                "Otherwise is_inappropriate=false and echo the original reply in message."
            )
        ),
        HumanMessage(content=message),
    ]
    output = grader.invoke(messages)
    if output.is_inappropriate:
        return True, output.message
    return False, message


In [ ]:
async def chat(message, history, thread_id):
    """Gradio passes `history` for display; the graph uses checkpointer + thread_id — only the new user turn is sent."""
    try:
        blocked, user_text = input_guardrail(message)
    except Exception:
        log.exception("input_guardrail failed")
        return "Could not validate your message. Please try again.", thread_id

    if blocked:
        return user_text, thread_id

    if not thread_id:
        thread_id = str(uuid.uuid4())

    config = {"configurable": {"thread_id": thread_id}}

    try:
        result = await graph.ainvoke(
            {"messages": [HumanMessage(content=user_text)]},
            config=config,
        )
    except Exception:
        log.exception("graph.ainvoke failed")
        return "The assistant crashed while processing that. Please try again.", thread_id

    last = result["messages"][-1]
    content = getattr(last, "content", None) or ""

    try:
        is_bad, out = output_guardrail(content)
    except Exception:
        log.exception("output_guardrail failed")
        return content if content else "Received an empty response; please try again.", thread_id

    if is_bad:
        return out, thread_id
    return content, thread_id


In [ ]:
_thread_state = gr.State(value=None)
demo = gr.ChatInterface(
    chat,
    type="messages",
    additional_inputs=[_thread_state],
    additional_outputs=[_thread_state],
    title="Euclid Design Agency",
    description="Ask Toby (lead architect) to research and design systems; he can search the web, use a browser, and render architecture diagrams.",
)

In [ ]:
demo.launch()